In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F

### 1.Brands

In [0]:
catalog_name = 'ecommerce'

#Define Schema for the data file  #StructField(name, dataType, nullable)
brand_schema = StructType([
    StructField('brand_code', StringType(), False),
    StructField('brand_name', StringType(), True),
    StructField('category_code', StringType(), True)
])

raw_data_path = '/Volumes/ecommerce/source_data/raw/brands/*.csv'

#create a df
df = spark.read\
        .option('header', 'true')\
        .option('delimiter', ',')\
        .schema(brand_schema)\
        .csv(raw_data_path) 

# add metadata columns
df = df.withColumn('_source_file', F.col('_metadata.file_path'))\
       .withColumn('ingested_at', F.current_timestamp()) 

display(df.limit(5))   

# Write raw data to the Bronze layer 
df.write\
  .format('delta')\
  .option('mergeSchema', 'true')\
  .mode('overwrite')\
  .saveAsTable(f'{catalog_name}.bronze.brz_brands')

brand_code,brand_name,category_code,_source_file,ingested_at
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-11-16T18:25:10.651Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-11-16T18:25:10.651Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-11-16T18:25:10.651Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-11-16T18:25:10.651Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-11-16T18:25:10.651Z


### 2.Category

In [0]:
#define schema
category_schema = StructType([
    StructField('category_code', StringType(), False),
    StructField('category_name', StringType(), True)
])
raw_data_path = '/Volumes/ecommerce/source_data/raw/category/*.csv'

#create a df
df = spark.read\
  .option("header", "true")\
  .option("delimiter", ",")\
  .schema(category_schema)\
  .csv(raw_data_path)

# Add metadata columns
df = df.withColumn('_ingested_at', F.current_timestamp())\
       .withColumn('_source_file', F.col('_metadata.file_path'))

display(df.limit(5))

# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_category)
df.write\
  .format('delta')\
  .option('mergeSchema', 'true')\
  .mode('overwrite')\
  .saveAsTable(f'{catalog_name}.bronze.brz_category')

category_code,category_name,_ingested_at,_source_file
ce,Electronics,2026-11-16T18:44:42.368Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
app,Apparel,2026-11-16T18:44:42.368Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
hnk,Home & Kitchen,2026-11-16T18:44:42.368Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
bpc,Beauty & Personal Care,2026-11-16T18:44:42.368Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
bks,Books,2026-11-16T18:44:42.368Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv


### 3.Products

In [0]:
#define schema
products_schema = StructType([
    StructField('product_id', StringType(), False),
    StructField('sku', StringType(), True),
    StructField('category_code', StringType(), True),
    StructField('brand_code', StringType(), True),
    StructField('color', StringType(), True),
    StructField('size', StringType(), True),
    StructField('material', StringType(), True),
    StructField('weight_grams', StringType(), True),  #datatype is string due to incoming data contain anamolies
    StructField('length_cm', StringType(), True),     #datatype is string due to incoming data contain anamolies
    StructField('width_cm', FloatType(), True),
    StructField('height_cm', FloatType(), True),
    StructField('rating_count', IntegerType(), True),
    StructField('file_name', StringType(), False),
    StructField('ingest_timestamp', TimestampType(), False)
])

#Load data using the schema defined
raw_data_path = '/Volumes/ecommerce/source_data/raw/products/*.csv'

#create df and metadata columns
df = spark.read\
  .option("header", "true")\
  .option("delimiter", ",")\
  .schema(products_schema)\
  .csv(raw_data_path) \
  .withColumn("file_name", F.col("_metadata.file_path")) \
  .withColumn("ingest_timestamp", F.current_timestamp())

# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_products)
df.write\
  .format('delta')\
  .option('mergeSchema', 'true')\
  .mode('overwrite')\
  .saveAsTable(f"{catalog_name}.bronze.brz_products")

### 4.Customers

In [0]:
#define schema
customers_schema = StructType([
    StructField('customer_id', StringType(), False),
    StructField('phone', StringType(), True),
    StructField('country_code', StringType(), True),
    StructField('country', StringType(), True),
    StructField('state', StringType(), True)
])

#Load data using the schema defined
raw_data_path = '/Volumes/ecommerce/source_data/raw/customers/*.csv'

#create df and metadata columns
df = spark.read\
  .option("header", "true")\
  .option("delimiter", ",")\
  .schema(customers_schema)\
  .csv(raw_data_path) \
  .withColumn("file_name", F.col("_metadata.file_path")) \
  .withColumn("ingest_timestamp", F.current_timestamp())

# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_customers)
df.write\
  .format('delta')\
  .option('mergeSchema', 'true')\
  .mode('overwrite')\
  .saveAsTable(f"{catalog_name}.bronze.brz_customers")

### 5.Date

In [0]:
# Define schema for the data file
date_schema = StructType([
    StructField('date', StringType(), True),           # Raw date in string format
    StructField('year', IntegerType(), True),          # Year
    StructField('day_name', StringType(), True),       # Day name (can be mixed case)
    StructField('quarter', IntegerType(), True),       # Quarter
    StructField('week_of_year', IntegerType(), True)   # Week of year (can be negative)
])

# Load data using the schema defined
raw_data_path = '/Volumes/ecommerce/source_data/raw/date/*.csv'

df_raw = spark.read.option("header", "true").option("delimiter", ",").schema(date_schema).csv(raw_data_path)

# Add metadata columns
df_raw = df_raw.withColumn("_ingested_at", F.current_timestamp()) \
               .withColumn("_source_file", F.col("_metadata.file_path"))

# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_calendar)
df_raw.write \
  .format("delta") \
  .option("mergeSchema", "true") \
  .mode("overwrite") \
  .saveAsTable(f"{catalog_name}.bronze.brz_calendar")